## Orbital Debris Exploration

Initial exploration of the Orbital Debris SQL database previously created. Contains various panda queries, sql queries, and visualizations to really 'dig' into the dataset.  This will help us narrow down and refine our primary and secondary questions, as well as help us polish the sql queries and chart visualizations for our primary and secondary questions.  Many of these query results can and likely will be used for supporting visuals to add context to our primary and secondary questions.  You may be able to tell a story in 3 visuals, but I am betting that story will be a broken story at best.

In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import utility as utils
import sqlite3 as sql
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from IPython.display import display, Markdown 

pd.set_option('display.max_columns', None)

# Initialize Visualization Settings
# Set the Palette (High Contrast Neon)
sns.set_palette("plasma") 

# Set the Background (Deep Space Black)
plt.style.use('dark_background')

plt.rcParams.update({
    "grid.alpha": 0.2,            # Faint, non-intrusive grid
    "axes.facecolor": "black",    # No grey haze in plot area
    "figure.facecolor": "black",  # No grey haze in outer area
    "text.color": "white",        # High contrast text
    "axes.labelcolor": "white",   # Axis labels
    "xtick.color": "white",       # Ticks
    "ytick.color": "white",
    "legend.facecolor": "black",  # Legend background
    "legend.edgecolor": "white"   # Legend border
})

print(plt.colormaps())

orbital_debris_conn = sql.connect('../data/clean/orbital_debris.db')

# Load kinetic_master as a pandas DataFrame for any eda we want to do using just pandas. We 
# could also load each table as a separate DataFrame, but this is easier for now and I can 
# use SQL queries against the proper database connection if I want to do more complex queries.
# Loading master_df just allows me to do quick things with pandas without having to write the
# SQL query logic each time.
master_df = pd.read_csv('../data/clean/kinetic_master.csv', low_memory=False)

display(utils.query_all_satellites(orbital_debris_conn))

['magma', 'inferno', 'plasma', 'viridis', 'cividis', 'twilight', 'twilight_shifted', 'turbo', 'berlin', 'managua', 'vanimo', 'Blues', 'BrBG', 'BuGn', 'BuPu', 'CMRmap', 'GnBu', 'Greens', 'Greys', 'OrRd', 'Oranges', 'PRGn', 'PiYG', 'PuBu', 'PuBuGn', 'PuOr', 'PuRd', 'Purples', 'RdBu', 'RdGy', 'RdPu', 'RdYlBu', 'RdYlGn', 'Reds', 'Spectral', 'Wistia', 'YlGn', 'YlGnBu', 'YlOrBr', 'YlOrRd', 'afmhot', 'autumn', 'binary', 'bone', 'brg', 'bwr', 'cool', 'coolwarm', 'copper', 'cubehelix', 'flag', 'gist_earth', 'gist_gray', 'gist_heat', 'gist_ncar', 'gist_rainbow', 'gist_stern', 'gist_yarg', 'gnuplot', 'gnuplot2', 'gray', 'hot', 'hsv', 'jet', 'nipy_spectral', 'ocean', 'pink', 'prism', 'rainbow', 'seismic', 'spring', 'summer', 'terrain', 'winter', 'Accent', 'Dark2', 'Paired', 'Pastel1', 'Pastel2', 'Set1', 'Set2', 'Set3', 'tab10', 'tab20', 'tab20b', 'tab20c', 'grey', 'gist_grey', 'gist_yerg', 'Grays', 'magma_r', 'inferno_r', 'plasma_r', 'viridis_r', 'cividis_r', 'twilight_r', 'twilight_shifted_r', 't

,norad_id,cospar_id,object_name,launch_year,launch_date,launch_site,owner,country_operator,object_type
0,5,1958-002B,VANGUARD 1,1958,1958-03-17,AFETR,US,USA,PAYLOAD
1,11,1959-001A,VANGUARD 2,1959,1959-02-17,AFETR,US,USA,PAYLOAD
2,12,1959-001B,VANGUARD R/B,1959,1959-02-17,AFETR,US,USA,ROCKET BODY
3,16,1958-002A,VANGUARD R/B,1958,1958-03-17,AFETR,US,USA,ROCKET BODY
4,20,1959-007A,VANGUARD 3,1959,1959-09-18,AFETR,US,USA,PAYLOAD
...,...,...,...,...,...,...,...,...,...
33351,68376,2026-009E,CZ-2C DEB,2026,2026-01-15,JSC,PRC,CHINA,DEBRIS
33352,68377,2026-062A,SUPERVIEW NEO-2 OBJECT A,2026,2026-03-25,TAISC,PRC,CHINA,PAYLOAD
33353,68378,2026-062B,SUPERVIEW NEO-2 OBJECT B,2026,2026-03-25,TAISC,PRC,CHINA,PAYLOAD
33354,68379,2026-062C,CZ-2D R/B,2026,2026-03-25,TAISC,PRC,CHINA,ROCKET BODY


### Payload Vulnerability Report (Pre-Zombie Check)
 
**The issue:**
A satellite’s operational status only tells us if it’s working right now—not whether it’s past its expected design life. Satellites that are still "operational" but older than 15 years are at higher risk of sudden failure and loss of control.

**What we do:**
- Identify all in-orbit payloads marked as `OPERATIONAL`.
- Flag those that are older than 15 years as "vulnerable" assets.
- Report the total number and share of these high-risk payloads, and break down the results by orbit class.

**Why it matters:**
Spotting aging, at-risk satellites helps us understand the true health of the active fleet and anticipate future risks. This is a first-pass estimate; a more detailed analysis will use actual design-life data in the next pipeline stage.

In [2]:
# Define the Vulnerability Threshold (Industry Standard: 5 - 15 Years)
vulnerability_threshold = 15

# Isolate Operational Payloads
op_payloads_mask = (master_df['in_orbit'] == 1) & \
                   (master_df['object_type'] == 'PAYLOAD') & \
                   (master_df['ops_status'] == 'OPERATIONAL')

op_payloads = master_df[op_payloads_mask].copy()

# Calculate Vulnerability
total_op = len(op_payloads)
vulnerable_op = op_payloads[op_payloads['sat_age_years'] > vulnerability_threshold]
vulnerable_count = len(vulnerable_op)

print(f"{'--- PAYLOAD VULNERABILITY AUDIT ---':^55}")
print(f"Total Operational Payloads:     {total_op:,}")
print(f"Aged Assets (>15 Years):        {vulnerable_count:,}")
print(f"Vulnerability Rate:             {(vulnerable_count/total_op if total_op > 0 else 0):.1%}")
print("-" * 55)

# Breakdown by Orbit Class
if vulnerable_count > 0:
    print("\nVulnerability Distribution by Orbit:")
    print(vulnerable_op['orbit_class'].value_counts())

print(f"\n👴 {vulnerable_count} high-risk active assets identified 👴")

          --- PAYLOAD VULNERABILITY AUDIT ---          
Total Operational Payloads:     14,791
Aged Assets (>15 Years):        394
Vulnerability Rate:             2.7%
-------------------------------------------------------

Vulnerability Distribution by Orbit:
orbit_class
LEO           194
GEO           148
MEO            26
UNKNOWN        16
ELLIPTICAL     10
Name: count, dtype: int64

👴 394 high-risk active assets identified 👴
